# Alternative Data Phase Test — Wheat Futures Direction Classification

**Goal:** Test each alternative data source with a simple BiGRU to see which improves prediction.

**Setup:** No Optuna. Same model for every test. Pure feature ablation.

| Test | Features |
|------|----------|
| Baseline | FRED-MD + Price |
| Phase 0 | + Technical indicators |
| Phase 1 | + CFTC COT positioning |
| Phase 2 | + Cross-commodity prices |
| All combined | Everything |

In [ ]:
# ============================================================
# SETUP
# ============================================================
import os, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except: pass

if IN_COLAB:
    BASE_DIR = '/content/drive/MyDrive/Quants'
    QUANTS_DIR = BASE_DIR  # FredMD_Dataset/ and Investing.com/ directly here
else:
    BASE_DIR = '/Users/np3129/Documents/AI_ML_Quants_VIP'
    QUANTS_DIR = os.path.join(BASE_DIR, 'Quants data', 'Quants ')

ALT_DATA_DIR = os.path.join(BASE_DIR, 'alternative_data', 'raw')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Bidirectional, GRU, Dense,
                                     GlobalAveragePooling1D, Dropout)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

print(f'Running on: {"Colab" if IN_COLAB else "Local"}')
print(f'Data: {QUANTS_DIR}')
print(f'Alt data: {ALT_DATA_DIR}')

## 1. Load Wheat Price Data

In [ ]:
inv_dir = os.path.join(QUANTS_DIR, 'Investing.com')
price_files = sorted([f for f in os.listdir(inv_dir) if f.endswith('.csv') and 'Wheat' in f])
print(f'Price files: {price_files}')

dfs = [pd.read_csv(os.path.join(inv_dir, f)) for f in price_files]
wheat = pd.concat(dfs, ignore_index=True)
wheat['Date'] = pd.to_datetime(wheat['Date'], format='mixed', dayfirst=False)
wheat = wheat.sort_values('Date').drop_duplicates(subset='Date').reset_index(drop=True)

for col in ['Price', 'Open', 'High', 'Low']:
    wheat[col] = pd.to_numeric(wheat[col].astype(str).str.replace(',', ''), errors='coerce')

def parse_vol(v):
    v = str(v).strip().replace(',', '')
    if v.endswith('K'): return float(v[:-1]) * 1000
    elif v.endswith('M'): return float(v[:-1]) * 1e6
    try: return float(v)
    except: return np.nan

wheat['Volume'] = wheat['Vol.'].apply(parse_vol)
wheat = wheat.dropna(subset=['Price']).reset_index(drop=True)
print(f'Wheat: {len(wheat)} days, {wheat["Date"].min().date()} to {wheat["Date"].max().date()}')

## 2. Load & Transform FRED-MD

In [ ]:
FRED_FEATURES = [
    'RPI','W875RX1','CMRMTSPLx','IPFPNSS','USWTRADE','USTRADE','BUSLOANS','CONSPI',
    'S&P 500','S&P PE ratio','FEDFUNDS','TB3MS','TB6MS','GS1','GS5','GS10','AAA','BAA',
    'TB3SMFFM','TB6SMFFM','T1YFFM','T5YFFM','T10YFFM','AAAFFM','BAAFFM',
    'EXSZUSx','EXJPUSx','EXUSUKx','EXCAUSx','PPICMM','UMCSENTx'
]

fred_raw = pd.read_csv(os.path.join(QUANTS_DIR, 'FredMD_Dataset', '2025-10-MD.csv'))
tcodes = fred_raw.iloc[0]
fred_data = fred_raw.iloc[1:].copy()
fred_data['sasdate'] = pd.to_datetime(fred_data['sasdate'])
fred_data = fred_data.rename(columns={'sasdate': 'Date'})

available = [f for f in FRED_FEATURES if f in fred_data.columns]
fred_data = fred_data[['Date'] + available].copy()
for col in available:
    fred_data[col] = pd.to_numeric(fred_data[col], errors='coerce')

tcode_map = {}
for f in available:
    try: tcode_map[f] = int(float(tcodes.get(f, 1)))
    except: tcode_map[f] = 1

def apply_tcode(series, tc):
    if tc == 1: return series
    elif tc == 2: return series.diff()
    elif tc == 3: return series.diff().diff()
    elif tc == 4: return np.log(series.clip(lower=1e-10))
    elif tc == 5: return np.log(series.clip(lower=1e-10)).diff()
    elif tc == 6: return np.log(series.clip(lower=1e-10)).diff().diff()
    return series

for col in available:
    fred_data[col] = apply_tcode(fred_data[col], tcode_map[col])

fred_data = fred_data.dropna().reset_index(drop=True)
fred_data['Date'] = fred_data['Date'] + pd.DateOffset(months=1)
fred_daily = fred_data.set_index('Date').resample('D').ffill().reset_index()
fred_cols = [c for c in fred_daily.columns if c != 'Date']
print(f'FRED-MD: {len(fred_cols)} features, {len(fred_daily)} daily rows')

## 3. Technical Indicators (Phase 0)

In [ ]:
def compute_tech(df):
    out = pd.DataFrame(index=df.index)
    c, h, l, v = df['Price'], df['High'], df['Low'], df['Volume'].fillna(0)
    for n in [5, 10, 20]:
        out[f'mom_{n}d'] = c.pct_change(n)
    d = c.diff()
    g, lo = d.clip(lower=0), (-d).clip(lower=0)
    out['rsi'] = 100 - (100 / (1 + g.rolling(14).mean() / lo.rolling(14).mean().replace(0, np.nan)))
    e12, e26 = c.ewm(span=12).mean(), c.ewm(span=26).mean()
    macd = e12 - e26
    out['macd'] = macd
    out['macd_hist'] = macd - macd.ewm(span=9).mean()
    s20, st20 = c.rolling(20).mean(), c.rolling(20).std()
    out['bb_pctb'] = (c - (s20 - 2*st20)) / (4*st20).replace(0, np.nan)
    tr = pd.concat([h-l, (h-c.shift(1)).abs(), (l-c.shift(1)).abs()], axis=1).max(axis=1)
    out['atr'] = tr.rolling(14).mean()
    out['vol_sma'] = v / v.rolling(20).mean().replace(0, np.nan)
    return out

tech = compute_tech(wheat)
tech_cols = tech.columns.tolist()
wheat = pd.concat([wheat, tech], axis=1)
print(f'Technical indicators: {tech_cols}')

## 4. CFTC COT Data (Phase 1)

In [ ]:
cot_raw = pd.read_csv(os.path.join(ALT_DATA_DIR, 'cot_wheat_disaggregated.csv'))
cot_raw['Date'] = pd.to_datetime(cot_raw['Report_Date_as_YYYY-MM-DD'])
nc = ['Open_Interest_All', 'Prod_Merc_Positions_Long_All', 'Prod_Merc_Positions_Short_All',
      'M_Money_Positions_Long_All', 'M_Money_Positions_Short_All']
for c in nc:
    cot_raw[c] = pd.to_numeric(cot_raw[c], errors='coerce')

cot = cot_raw.groupby('Date')[nc].sum().reset_index()
oi = cot['Open_Interest_All'].replace(0, np.nan)
cot['mm_net'] = cot['M_Money_Positions_Long_All'] - cot['M_Money_Positions_Short_All']
cot['cm_net'] = cot['Prod_Merc_Positions_Long_All'] - cot['Prod_Merc_Positions_Short_All']
cot['hedge_pr'] = cot['cm_net'] / oi
cot['spec_se'] = cot['mm_net'] / oi
cot['pos_chg'] = cot['mm_net'].diff(4)
cot['spec_z'] = (cot['mm_net'] - cot['mm_net'].rolling(52).mean()) / cot['mm_net'].rolling(52).std().replace(0, np.nan)
cot['oi_lev'] = cot['Open_Interest_All']
cot_features = ['hedge_pr', 'spec_se', 'pos_chg', 'spec_z', 'oi_lev']

cot['Date'] = cot['Date'] + pd.Timedelta(days=3)  # publication delay
cot_daily = cot[['Date'] + cot_features].set_index('Date').resample('D').ffill().reset_index()
print(f'COT: {len(cot_daily)} daily rows, features={cot_features}')

## 5. Cross-Commodity Prices (Phase 2)

In [ ]:
cross_assets = {'corn_futures.csv':'corn', 'soybean_futures.csv':'soy',
                'crude_oil_wti.csv':'oil', 'usd_index.csv':'usd', 'gold_futures.csv':'gold'}
cdfs = []
for fn, lb in cross_assets.items():
    fp = os.path.join(ALT_DATA_DIR, fn)
    # yfinance v2 CSVs have multi-level headers
    d = pd.read_csv(fp, header=[0, 1], index_col=0, parse_dates=True)
    d.columns = [c[0] for c in d.columns]  # flatten
    d = d.reset_index()
    d['Date'] = pd.to_datetime(d['Date'])
    d[f'{lb}_cl'] = pd.to_numeric(d['Close'], errors='coerce')
    cdfs.append(d[['Date', f'{lb}_cl']].dropna())
    print(f'  {lb}: {len(d)} rows')

cr = cdfs[0]
for d in cdfs[1:]:
    cr = pd.merge(cr, d, on='Date', how='outer')
cr = cr.sort_values('Date').ffill().dropna().reset_index(drop=True)

# Compute features using wheat price
cr = pd.merge(cr, wheat[['Date', 'Price']].rename(columns={'Price': 'w_cl'}), on='Date', how='inner')
cr['wc_rat'] = cr['w_cl'] / cr['corn_cl'].replace(0, np.nan)
cr['ws_rat'] = cr['w_cl'] / cr['soy_cl'].replace(0, np.nan)
cr['wc_m20'] = cr['wc_rat'].pct_change(20)
cr['ws_m20'] = cr['ws_rat'].pct_change(20)
wr = cr['w_cl'].pct_change(10)
cr['rel_cn'] = wr - cr['corn_cl'].pct_change(10)
cr['rel_ol'] = wr - cr['oil_cl'].pct_change(10)
cr['usd_c10'] = cr['usd_cl'].pct_change(10)
cr['gld_c10'] = cr['gold_cl'].pct_change(10)
cross_fcols = ['wc_rat', 'ws_rat', 'wc_m20', 'ws_m20', 'rel_cn', 'rel_ol', 'usd_c10', 'gld_c10']
cr = cr.drop(columns=['w_cl'])
print(f'Cross-commodity: {len(cr)} rows, features={cross_fcols}')

## 6. Merge All Data

In [ ]:
# merge_asof handles date mismatches gracefully
master = wheat[['Date', 'Price'] + tech_cols].copy().sort_values('Date')
master = pd.merge_asof(master, fred_daily.sort_values('Date'), on='Date', direction='backward')
master = pd.merge_asof(master, cot_daily.sort_values('Date'), on='Date', direction='backward')
master = pd.merge_asof(master, cr[['Date'] + cross_fcols].sort_values('Date'), on='Date', direction='backward')

master = master[master['Date'] >= '2013-06-01'].reset_index(drop=True)
master = master.ffill().bfill()
master = master.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

# Target
master['target'] = (master['Price'].shift(-1) > master['Price']).astype(int)
master = master.iloc[:-1].reset_index(drop=True)

print(f'Master: {master.shape}')
print(f'Range: {master["Date"].min().date()} to {master["Date"].max().date()}')
print(f'Target: Up={master["target"].mean():.1%}, Down={1-master["target"].mean():.1%}')
print(f'\nFeature counts: FRED={len(fred_cols)}, Tech={len(tech_cols)}, COT={len(cot_features)}, Cross={len(cross_fcols)}, Price=1')

## 7. Define Experiments

In [ ]:
base = fred_cols + ['Price']

EXPERIMENTS = {
    'Baseline (FRED+Price)':   base,
    'Phase 0 (+Tech)':         base + tech_cols,
    'Phase 1 (+COT)':          base + cot_features,
    'Phase 2 (+Cross)':        base + cross_fcols,
    'Ph0+1 (Tech+COT)':        base + tech_cols + cot_features,
    'ALL COMBINED':            base + tech_cols + cot_features + cross_fcols,
}

for name, feats in EXPERIMENTS.items():
    print(f'{name:30s} -> {len(feats)} features')

## 8. Model & Runner

In [ ]:
LOOKBACK = 30
EPOCHS = 50
BATCH = 64

def make_windows(df, fcols, lb=30):
    data, tgt = df[fcols].values, df['target'].values
    X, y = [], []
    for i in range(lb, len(data)):
        X.append(data[i-lb:i])
        y.append(tgt[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def build_bigru(seq_len, n_feat, hidden=64, do=0.2, wd=0.01):
    inp = Input(shape=(seq_len, n_feat))
    x = Bidirectional(GRU(hidden, return_sequences=True, kernel_regularizer=l2(wd)))(inp)
    x = Dropout(do)(x)
    x = Bidirectional(GRU(hidden, return_sequences=True, kernel_regularizer=l2(wd)))(x)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(do)(x)
    out = Dense(1, activation='sigmoid', kernel_regularizer=l2(wd))(x)
    m = Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy', metrics=['accuracy'])
    return m

def run_experiment(name, fcols, master_df, lookback=LOOKBACK, epochs=EPOCHS, batch=BATCH):
    print(f'\n{"="*55}')
    print(f'  {name} ({len(fcols)} features)')
    print(f'{"="*55}')
    
    split_idx = int(len(master_df) * 0.8)
    
    # Scale on train only
    scaler = StandardScaler()
    scaler.fit(master_df.iloc[:split_idx][fcols])
    scaled = master_df.copy()
    scaled[fcols] = scaler.transform(master_df[fcols])
    scaled[fcols] = scaled[fcols].replace([np.inf, -np.inf], 0).fillna(0)
    
    X, y = make_windows(scaled, fcols, lookback)
    te = split_idx - lookback
    Xtr, ytr = X[:te], y[:te]
    Xte, yte = X[te:], y[te:]
    print(f'  Train: {Xtr.shape}, Test: {Xte.shape}')
    
    cw = dict(zip([0, 1], compute_class_weight('balanced', classes=np.array([0, 1]), y=ytr)))
    
    tf.keras.backend.clear_session()
    model = build_bigru(lookback, len(fcols))
    model.fit(Xtr, ytr, epochs=epochs, batch_size=batch, class_weight=cw, verbose=0,
              callbacks=[EarlyStopping(monitor='loss', patience=7, restore_best_weights=True),
                         ReduceLROnPlateau(monitor='loss', factor=0.5, patience=4, min_lr=1e-6)])
    
    yp = model.predict(Xte, verbose=0).flatten()
    ypred = (yp >= 0.5).astype(int)
    
    r = {
        'Experiment': name, 'Features': len(fcols),
        'Accuracy': accuracy_score(yte, ypred),
        'Precision': precision_score(yte, ypred, zero_division=0),
        'Recall': recall_score(yte, ypred, zero_division=0),
        'F1': f1_score(yte, ypred, zero_division=0),
        'AUC-ROC': roc_auc_score(yte, yp),
    }
    print(f'  Acc={r["Accuracy"]:.4f}  F1={r["F1"]:.4f}  AUC={r["AUC-ROC"]:.4f}')
    return r, yte, ypred, yp

## 9. Run ALL Experiments

In [ ]:
all_results = []
all_preds = {}

for name, fcols in EXPERIMENTS.items():
    r, yte, ypred, yprob = run_experiment(name, fcols, master)
    all_results.append(r)
    all_preds[name] = {'y_test': yte, 'y_pred': ypred, 'y_prob': yprob}

print('\n\nAll experiments complete!')

## 10. Results

In [ ]:
results_df = pd.DataFrame(all_results).set_index('Experiment')

print('=' * 70)
print('RESULTS — Which Alternative Data Helps?')
print('=' * 70)
print(results_df.to_string(float_format='{:.4f}'.format))
print()
print(f'Best Accuracy: {results_df["Accuracy"].idxmax()} ({results_df["Accuracy"].max():.4f})')
print(f'Best AUC-ROC:  {results_df["AUC-ROC"].idxmax()} ({results_df["AUC-ROC"].max():.4f})')
print(f'Best F1:       {results_df["F1"].idxmax()} ({results_df["F1"].max():.4f})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics = ['Accuracy', 'AUC-ROC', 'F1']
colors = ['#54A0FF', '#00D4AA', '#FF9F43']

for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric]
    bars = ax.barh(vals.index, vals.values, color=color, alpha=0.85)
    ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Random')
    ax.set_xlabel(metric)
    ax.set_xlim(0.35, max(0.65, vals.max() + 0.03))
    ax.legend()
    for bar, val in zip(bars, vals.values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)

fig.suptitle('Phase Test Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name in zip(axes, ['Baseline (FRED+Price)', 'ALL COMBINED']):
    p = all_preds[name]
    cm = confusion_matrix(p['y_test'], p['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Down', 'Up'], yticklabels=['Down', 'Up'])
    ax.set_title(name); ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.suptitle('Confusion Matrix: Baseline vs All Combined', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
out_path = os.path.join(BASE_DIR, 'alternative_data', 'processed', 'phase_test_results.csv')
os.makedirs(os.path.dirname(out_path), exist_ok=True)
results_df.to_csv(out_path)
print(f'Results saved to: {out_path}')